# PARC2026 — π0.5 smoke + GA=8 verification

このNotebookは、運営GPUで本学習を始める前の **Gate O1** をColab A100で潰すためのものです。

単体で実行できるよう、冒頭に `00_a100_preflight.ipynb` 相当のworkspace/repo準備を内包します。運営combined datasetがColabに無い場合は、公開 `Sylvest/libero_plus_lerobot` をColab一時領域へ取得して smoke/GA 検証を進めます。公開datasetでのPASSは学習pipeline/GAの確認であり、運営datasetでの最終再現を置き換えません。

## Self-contained preflight
`00_a100_preflight.ipynb` を別に実行していなくても、このセルでGPU確認・workspace作成・repo clone/updateまで行います。

In [ ]:
from pathlib import Path
import os, platform, shutil, subprocess, sys

print('python:', sys.version)
print('platform:', platform.platform())
subprocess.run(['nvidia-smi'], check=True)

ROOT = Path('/content/parc2026')
for p in [ROOT, ROOT/'vendor', ROOT/'cache', ROOT/'datasets', ROOT/'outputs']:
    p.mkdir(parents=True, exist_ok=True)
REPO = ROOT / 'py_AI'
if not (REPO / '.git').exists():
    subprocess.run(['git', 'clone', 'https://github.com/yu37330/py_AI.git', str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'fetch', '--all', '--prune'], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', 'main'], check=True)
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', 'main'], check=True)
PI05_DIR = REPO / 'examples/pi05_libero_finetune'
print('workspace:', ROOT)
print('repo:', REPO)
print('git:', subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'], text=True).strip())

## Python 3.10を用意
Colab runtimeのPython版に依存せず、`uv` でPython 3.10を用意します。

In [ ]:
if shutil.which('uv') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
subprocess.run(['uv', 'python', 'install', '3.10'], check=True)
PY310 = subprocess.check_output(['uv', 'python', 'find', '3.10'], text=True).strip()
print('python3.10:', PY310)
subprocess.run([PY310, '--version'], check=True)

## HF token
PaliGemma access権のあるtokenをNotebookへ保存せずに入力します。Colab Secretsに `HF_TOKEN` があればそれを優先します。

In [ ]:
from getpass import getpass
if not os.environ.get('HF_TOKEN'):
    try:
        from google.colab import userdata
        token = userdata.get('HF_TOKEN')
    except Exception:
        token = None
    if token:
        os.environ['HF_TOKEN'] = token
    else:
        os.environ['HF_TOKEN'] = getpass('HF token (PaliGemma access権があるtoken): ')
print('HF_TOKEN: set (not displayed)')

## Datasetを解決
優先順位は `PI05_DATASET_ROOT` → Colab上の運営 `libero_combined_20hz` → 公開LIBERO-plus fallback です。

公開fallbackは `meta/ + data/ + videos/` を取得するため容量・時間を使います。Colabの一時領域にのみ置き、Google Driveやgitには保存しません。

In [ ]:
PUBLIC_DATASET_ID = os.environ.get('PI05_PUBLIC_DATASET_ID', 'Sylvest/libero_plus_lerobot')
configured = os.environ.get('PI05_DATASET_ROOT')
organizer_root = ROOT / 'datasets' / 'libero_combined_20hz'

if configured:
    DATASET_ROOT = Path(configured)
    DATASET_REPO_ID = os.environ.get('PI05_DATASET_REPO_ID', 'local/libero_combined_20hz')
    DATASET_SOURCE = 'configured'
elif (organizer_root / 'meta' / 'info.json').exists():
    DATASET_ROOT = organizer_root
    DATASET_REPO_ID = os.environ.get('PI05_DATASET_REPO_ID', 'local/libero_combined_20hz')
    DATASET_SOURCE = 'organizer_combined'
else:
    PUBLIC_ROOT = ROOT / 'datasets' / 'public_libero_plus_lerobot'
    if not (PUBLIC_ROOT / 'meta' / 'info.json').exists():
        try:
            from huggingface_hub import snapshot_download
        except ImportError:
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub'], check=True)
            from huggingface_hub import snapshot_download
        print('organizer dataset not found; downloading public training dataset:', PUBLIC_DATASET_ID)
        print('This can be a large download. Files stay under /content/parc2026/datasets.')
        snapshot_download(
            repo_id=PUBLIC_DATASET_ID,
            repo_type='dataset',
            local_dir=str(PUBLIC_ROOT),
            allow_patterns=['meta/*', 'data/*', 'videos/*'],
            token=os.environ.get('HF_TOKEN'),
        )
    DATASET_ROOT = PUBLIC_ROOT
    DATASET_REPO_ID = PUBLIC_DATASET_ID
    DATASET_SOURCE = f'public:{PUBLIC_DATASET_ID}'

for required in ['meta', 'data', 'videos']:
    if not (DATASET_ROOT / required).exists():
        raise FileNotFoundError(f'{DATASET_ROOT / required} がありません')
print('dataset source:', DATASET_SOURCE)
print('dataset repo id:', DATASET_REPO_ID)
print('dataset root:', DATASET_ROOT)
if DATASET_SOURCE.startswith('public:'):
    print('NOTE: 公開LIBERO-plusでのsmokeです。運営combined datasetの最終再現とは分離して扱います。')

## 学習環境setup
venv / HF cache / outputsはColabの `/content` 側へ置きます。

In [ ]:
TRAIN_DATA_ROOT = ROOT / 'cache' / 'pi05-train'
LEROBOT_ROOT = ROOT / 'vendor' / 'lerobot-pi05'
TRAIN_DATA_ROOT.mkdir(parents=True, exist_ok=True)
env = os.environ.copy()
env.update({
    'PYTHON': PY310,
    'DATA_ROOT': str(TRAIN_DATA_ROOT),
    'LEROBOT_ROOT': str(LEROBOT_ROOT),
})
subprocess.run(['bash', 'scripts/setup_train.sh'], cwd=PI05_DIR, env=env, check=True)

## GA=8 runtime probe
最初は BS=1 / GA=8 / 3 optimizer stepだけ回します。期待値は **24 backward micro-step → 3 optimizer update** です。

In [ ]:
GA = 8
PROBE_STEPS = 3
PROBE_BS = 1
TRACE = ROOT / 'outputs' / 'pi05_ga8_probe_trace.jsonl'
TRACE_SUMMARY = ROOT / 'outputs' / 'pi05_ga8_probe_summary.json'
TRACE.parent.mkdir(parents=True, exist_ok=True)
TRACE.unlink(missing_ok=True)
TRACE_SUMMARY.unlink(missing_ok=True)
probe_env = os.environ.copy()
probe_env.update({
    'PARC_GA_TRACE_FILE': str(TRACE),
    'PYTHONPATH': str(REPO / 'tools/pi05/ga_instrument') + (':' + probe_env['PYTHONPATH'] if probe_env.get('PYTHONPATH') else ''),
    'PI05_DATASET_ROOT': str(DATASET_ROOT),
    'PI05_DATASET_REPO_ID': DATASET_REPO_ID,
    'PI05_VIDEO_BACKEND': os.environ.get('PI05_VIDEO_BACKEND', 'pyav'),
    'SMOKE_BS': str(PROBE_BS),
    'SMOKE_GA': str(GA),
    'SMOKE_STEPS': str(PROBE_STEPS),
    'SMOKE_SKIP_MERGE': '1',
    'RUN_NAME': 'colab_ga8_probe',
})
cmd = 'source env_train.sh && bash scripts/smoke_pi05.sh'
subprocess.run(['bash', '-lc', cmd], cwd=PI05_DIR, env=probe_env, check=True)
print('trace:', TRACE)

In [ ]:
summary_cmd = [
    sys.executable, str(REPO / 'tools/pi05/summarize_ga_trace.py'), str(TRACE),
    '--expected-ga', str(GA), '--expected-steps', str(PROBE_STEPS),
    '--json-out', str(TRACE_SUMMARY),
]
subprocess.run(summary_cmd, check=True)
print('GA Gate: PASS')
print('summary:', TRACE_SUMMARY)

## Optional: 20-step smoke + LoRA merge
GA Gate PASS後だけ `RUN_FULL_20_STEP=True` にして実施します。

In [ ]:
RUN_FULL_20_STEP = False
if RUN_FULL_20_STEP:
    full_env = os.environ.copy()
    full_env.update({
        'PI05_DATASET_ROOT': str(DATASET_ROOT),
        'PI05_DATASET_REPO_ID': DATASET_REPO_ID,
        'PI05_VIDEO_BACKEND': os.environ.get('PI05_VIDEO_BACKEND', 'pyav'),
        'SMOKE_BS': '2',
        'SMOKE_GA': '8',
        'SMOKE_STEPS': '20',
        'RUN_NAME': 'colab_pi05_smoke_ga8_20',
    })
    subprocess.run(['bash', '-lc', 'source env_train.sh && bash scripts/smoke_pi05.sh'], cwd=PI05_DIR, env=full_env, check=True)
    print('20-step smoke + merge: PASS')
else:
    print('skip: RUN_FULL_20_STEP=False')

## Exit criteria
必須Gateは `GA Gate: PASS` です。公開fallbackでPASSした場合もGA semanticsのEvidenceとして保存できますが、Run A固定前には運営combined datasetでも短い再確認を行います。